In [10]:
# We need to PIP install spaCY libraries. Uncommend the following lines to do so.
# pip install -U spacy
# We also need to download the English model for spaCY.
# python -m spacy download en_core_web_sm
# If you want a dutch library, just chantge the language code to nl_core_news_sm.

# Then we import the necessary libraries.
import pandas as pd
import os
import spacy
from spacy.lang.en.stop_words import STOP_WORDS

Now we read the bronze data, and import a small pipeline

In [11]:
# we can import a small pipeline that is used for tokenization, part-of-speech tagging, and named entity recognition.
nlp = spacy.load('en_core_web_sm')

#Now we dont have to make the pipeline ourselves, we can just use the nlp object to process our text data. 
# We will do this in the silver layer, but we can already load the bronze data here.
bronze_data = pd.read_csv('Data/bronze/bronze_data.csv')
print(bronze_data.head())

       id                                              title  \
0  doc_01  Assessing Climate Adaptation Measures in Low-L...   
1  doc_02  Evaluating Circular Economy Practices in Regio...   
2  doc_03  Data Integration Challenges in Cross-Sector Re...   
3  doc_04  Designing Indicators for Municipal Climate Res...   
4  doc_05  Improving Data Literacy Through Applied Educat...   

                                             content  
0  Introduction\nClimate adaptation has become a ...  
1  Introduction\nThe circular economy seeks to re...  
2  Introduction\nCross-sector research collaborat...  
3  Introduction\nMunicipalities play a crucial ro...  
4  Introduction\nData literacy has become an esse...  


now we can run the small pipeline on 1 of our rows in the dataset

In [12]:
# Here we will run ther NLP on 1 of the rows to see what it does.
# we leave out the identifier colum (ID), because NLP is only interested in the text data.
# So we will only pass the Title and Content columns through the NLP pipeline.
row = bronze_data.iloc[0]
text = f"{row['title']}\n\n{row['content']}"

# We did the "\n\n" to separate the title and content, so that the NLP pipeline can 
# recognize that they are different parts of the text. the double \n is used to create a whitespace between title and content.  
# Now we have a string that contains the title and content of the article. We can now pass this through the NLP pipeline.
doc = nlp(text)
print(doc.text)

Assessing Climate Adaptation Measures in Low-Lying Coastal Regions

Introduction
Climate adaptation has become a central topic in regional planning as coastal areas face increasing risks from sea-level rise, extreme precipitation, and prolonged droughts. Low-lying coastal regions are particularly vulnerable due to their exposure to flooding, saltwater intrusion, and land subsidence. These risks threaten not only physical infrastructure but also ecosystems, economic activities, and the livability of coastal communities. As climate impacts intensify, regional authorities are under growing pressure to implement effective adaptation measures.

Background and context
Over the past decade, a wide range of climate adaptation strategies have been proposed and implemented in coastal regions. These strategies include hard infrastructure solutions such as dikes, storm surge barriers, and pumping stations, as well as nature-based solutions like wetland restoration, dune reinforcement, and managed 

In [13]:
# Now we can play around with the doc object

for token in doc[:20]: # limit to the first 20 tokens
    print(token.text)
# This shows that the tokenization works, it splits the text into individual words and punctuation marks like "-".

Assessing
Climate
Adaptation
Measures
in
Low
-
Lying
Coastal
Regions



Introduction


Climate
adaptation
has
become
a
central
topic


In [14]:
# The doc.sents attribute gives us the sentences in the text. We can iterate over these sentences and print them out.
for sent in doc.sents:
    print("SENT:", sent.text)

# because the title or heading are not ending with a dot, the NLP pipeline does not recognize them as sentences. 

SENT: Assessing Climate Adaptation Measures in Low-Lying Coastal Regions

Introduction
Climate adaptation has become a central topic in regional planning as coastal areas face increasing risks from sea-level rise, extreme precipitation, and prolonged droughts.
SENT: Low-lying coastal regions are particularly vulnerable due to their exposure to flooding, saltwater intrusion, and land subsidence.
SENT: These risks threaten not only physical infrastructure but also ecosystems, economic activities, and the livability of coastal communities.
SENT: As climate impacts intensify, regional authorities are under growing pressure to implement effective adaptation measures.


SENT: Background and context
Over the past decade, a wide range of climate adaptation strategies have been proposed and implemented in coastal regions.
SENT: These strategies include hard infrastructure solutions such as dikes, storm surge barriers, and pumping stations, as well as nature-based solutions like wetland restorat

In [15]:
# we can also print the named entities in the text.
# Entities are things like people, organizations, locations, dates, etc. that are mentioned in the text.
for ent in doc.ents:
    print(ent.text, ent.label_)

Background GPE
the past decade DATE
Three CARDINAL
one CARDINAL
third ORDINAL
one CARDINAL


Now we got a feeling of what the spacy library can do.  
Now we will look at the important things, and start with pre-processing steps.  

In [16]:
# we load the bronze dataset in again, since we moddified it before.
bronze_data = pd.read_csv('Data/bronze/bronze_data.csv')

# First we will check if there is data with missing title or content and if there are rows where content is not a string.
bronze_data[["title","content"]].isna().sum()
bronze_data["content"].map(type).value_counts().head()

content
<class 'str'>    10
Name: count, dtype: int64

In [17]:
# we always check for duplicates
bronze_data.duplicated(subset=["title", "content"]).sum()

0

In [18]:
# we will check for boiler plate text such as headings or footers.
# These are not useful for our analysis and we want to remove them. We can do this by checking for common phrases that are used in these boiler plate texts.
# we will use the counter from collections to count the most common phrases in the content column.

from collections import Counter

def lines(text):
    return [line.strip() for line in text.splitlines() if line.strip()]

cnt = Counter()
for text in bronze_data["content"].dropna():
    cnt.update(lines(text))

print(cnt.most_common(10))


[('Introduction', 10), ('Problem description', 10), ('Discussion', 10), ('Recommendations', 10), ('Conclusion', 10), ('Future work', 10), ('Objectives', 9), ('Methodology', 6), ('Research questions', 5), ('Findings', 5)]
